# Notebook para ejercicios 4, 5 y 6

In [77]:
import Pkg; Pkg.add("SymDoME")

   Resolving package versions...
   Installed SymDoME ─ v1.0.2
    Updating `~/.julia/environments/v1.11/Project.toml`
  [b4f4f0b4] + SymDoME v1.0.2
    Updating `~/.julia/environments/v1.11/Manifest.toml`
  [b4f4f0b4] + SymDoME v1.0.2
Precompiling project...
   5858.3 ms  ✓ SymDoME
  1 dependency successfully precompiled in 12 seconds. 506 already precompiled.


In [120]:
include("/Users/rafa/Documents/Uni/Cuatri4/machine_learning/FAA_practices/src/boletin_1.2/45159263M_49918198X_48118738R_54153358L.jl")

ANNCrossValidation (generic function with 1 method)

## Ejercicio 4

In [19]:
function confusionMatrix(outputs::AbstractArray{Bool,1}, targets::AbstractArray{Bool,1})

    true_positives = sum(outputs .& targets)
    false_positives = sum(outputs .& .!targets)
    true_negatives = sum(.!outputs .& .!targets)
    false_negatives = sum(.!outputs .& targets)

    accuracy = (true_positives + true_negatives) / length(outputs) # precision
    fail_rate = (false_positives + false_negatives) / length(outputs) # tasa de fallo
    recall = (true_positives == 0 && false_negatives == 0) ? 1 : true_positives / (true_positives + false_negatives) # sensibilidad
    especificity = (true_negatives == 0 && false_positives == 0) ? 1 : true_negatives / (true_negatives + false_positives) # especificidad
    precision = (true_positives == 0 && false_positives == 0) ? 1 : true_positives / (true_positives + false_positives) # valor predictivo positivo
    npv = (true_negatives == 0 && false_negatives == 0) ? 1 : true_negatives / (true_negatives + false_negatives) # valor predictivo negativo
    f1 = (recall == 0 && precision == 0) ? 0 : 2 * (precision * recall) / (precision + recall) # f1 score
    confussion_matrix = [true_negatives false_positives; false_negatives true_positives] # matriz de confusión

    return (accuracy, fail_rate, recall, especificity, precision, npv, f1, confussion_matrix)
end

confusionMatrix (generic function with 1 method)

In [20]:
function confusionMatrix(outputs::AbstractArray{<:Real,1},
    targets::AbstractArray{Bool,1}; threshold::Real=0.5)

    outputs = outputs .> threshold
    return confusionMatrix(outputs, targets)

end

confusionMatrix (generic function with 2 methods)

In [21]:
function confusionMatrix(outputs::AbstractArray{Bool,2},
    targets::AbstractArray{Bool,2}; weighted::Bool=true)

    num_classes = size(targets, 2)

    if size(outputs, 2) == size(targets, 2) && size(outputs, 2) > 2

        recall = zeros(num_classes) # Guardar memoria para sensibilidad
        especificity = zeros(num_classes) # Guardar memoria para especificidad
        precision = zeros(num_classes) # Guardar memoria para valor predictivo positivo
        npv = zeros(num_classes) # Guardar memoria para valor predictivo negativo
        f1 = zeros(num_classes) # Guardar memoria para f1 score

        for i in 1:size(outputs, 2)
            recall[i], especificity[i], precision[i], npv[i], f1[i], _ = confusionMatrix(outputs[:, i], targets[:, i])
        end

        confussion_matrix = [sum(outputs[:, i] .& targets[:, j]) for i in 1:num_classes, j in 1:num_classes] # Matriz de confusion

        if weighted
            w = vec(sum(targets, dims=1))/size(targets, 1) # Peso de cada clase
        else
            w = repeat([1/num_classes], num_classes) # Peso uniforme (= media aritmética)
        end

        # Calcular métricas según el peso de cada clase (weighted o media aritmética) 
        recall = sum(recall .* w)
        especificity = sum(especificity .* w)
        precision = sum(precision .* w)
        npv = sum(npv .* w)
        f1 = sum(f1 .* w)
        acc = accuracy(outputs, targets)
        fail_rate = 1 - acc

        return(acc, fail_rate, recall, especificity, precision, npv, f1, confussion_matrix)


    elseif size(outputs, 2) == size(targets, 2) && size(outputs, 2) == 1 # Clasificacion binaria
        return confusionMatrix(outputs[:], targets[:])
    end
end

confusionMatrix (generic function with 3 methods)

In [22]:
function confusionMatrix(outputs::AbstractArray{<:Real,2},
    targets::AbstractArray{Bool,2}; threshold::Real=0.5, weighted::Bool=true)

    outputs = classifyOutputs(outputs; threshold=threshold)
    return confusionMatrix(outputs, targets, weighted=weighted)

end
    

confusionMatrix (generic function with 4 methods)

In [24]:
function confusionMatrix(outputs::AbstractArray{<:Any,1},
    targets::AbstractArray{<:Any,1},
    classes::AbstractArray{<:Any,1}; weighted::Bool=true)

    @assert(all([in(label, classes) for label in vcat(targets, outputs)])) # Comprobar que las etiquetas son correctas
    @assert size(outputs, 1) == size(targets, 1)

    outputs = oneHotEncoding(outputs, classes)
    targets = oneHotEncoding(targets, classes)

    return confusionMatrix(outputs, targets, weighted=weighted)
end
    

confusionMatrix (generic function with 6 methods)

In [23]:
function confusionMatrix(outputs::AbstractArray{<:Any,1},
    targets::AbstractArray{<:Any,1}; weighted::Bool=true)

    classes = unique(vcat(outputs, targets))
    return confusionMatrix(outputs, targets, classes, weighted=weighted)
end

confusionMatrix (generic function with 5 methods)

## Ejercicio 5

In [25]:
using Random

### Crossvalidation

In [26]:
function crossvalidation(N::Int64, k::Int64)

    folds = 1:k # number of folds
    k_folds = repeat(folds, Int(ceil(N/k))) # number of elements in each fold
    k_folds = k_folds[1:N] # remove the extra elements
    n_folds = shuffle!(k_folds) # shuffle the elements in each fold
    return n_folds

end
    

crossvalidation (generic function with 1 method)

In [27]:
# Clasificacion binaria

function crossvalidation(targets::AbstractArray{Bool,1}, k::Int64)

   # Asegurarse que cada clase tiene al menos 10 representantes
   min_class_count = min(sum(targets), sum(.!targets))
   if min_class_count < 10     
      return
   end
    
   indices = collect(1:length(targets))
   indices[targets] = crossvalidation(sum(targets), k) # asignar a cada fila un valor de la lista de folds
   indices[.!targets] = crossvalidation(sum(.!targets), k) # Llamar a la funcion anterior con el numero de instancias negativas
   return indices
   
end


crossvalidation (generic function with 2 methods)

In [28]:
# Clasificacion multiclase

function crossvalidation(targets::AbstractArray{Bool,2}, k::Int64)

    # Asegurarse que cada calse tiene al menos 10 representantes
    class_counts = vec(sum(targets, dims=1))  # Número de instancias por clase

    min_class_count = minimum(class_counts)  # Mínimo de instancias en cualquier clase
    if min_class_count < 10
        return
    end

    indices = collect(1:size(targets, 1))
    for i in 1:size(targets, 2)
        indices[targets[:,i]] = crossvalidation(sum(targets[:,i]), k) # asignar a cada fila un valor de la lista de folds
    end

    return indices
end

crossvalidation (generic function with 3 methods)

In [29]:
function crossvalidation(targets::AbstractArray{<:Any,1}, k::Int64) 

    return crossvalidation(oneHotEncoding(targets), k)

end

crossvalidation (generic function with 4 methods)

### Train a K-fold ANN

In [ ]:
function ANNCrossValidation(topology::AbstractArray{<:Int,1},
    dataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{<:Any,1}},
    crossValidationIndices::Array{Int64,1};
    numExecutions::Int=50,
    transferFunctions::AbstractArray{<:Function,1}=fill(σ, length(topology)),
    maxEpochs::Int=1000, minLoss::Real=0.0, learningRate::Real=0.01,
    validationRatio::Real=0, maxEpochsVal::Int=20) 

    inputs, targets = dataset # Descomponer dataset
    classes = unique(targets) # Calcular las clases
    one_hot = oneHotEncoding(targets, classes) # OneHot de las clases

    folds = maximum(crossValidationIndices) # Calcular el número de folds


    accuracy = Float64[] # Vector de precisión (accuracy)
    fail_rate = Float64[] # Vector de tasa de error (error rate)
    recall = Float64[] # Vector de sensibilidad (recall)
    especificity = Float64[] # Vector de especificidad
    precision = Float64[] # Vector de VPP (precisión)
    npv = Float64[] # Vector de VPN
    f1 = Float64[] # Vector de F1
    confussion_matrix = zeros(length(classes), length(classes)) # Inicializar confussion matrix

    for fold in 1:folds

        # Extraer datos de entrenamiento y test según folds
        train_inputs = inputs[findall(crossValidationIndices .!= fold), :]
        train_targets = one_hot[findall(crossValidationIndices .!= fold), :]
        test_inputs = inputs[findall(crossValidationIndices .== fold), :]
        test_targets = one_hot[findall(crossValidationIndices .== fold), :]

        validation_inputs = Matrix{Float32}(undef, 0, size(train_inputs, 2)) # Inicializar validation inputs como matriz vacía
        validation_targets = Matrix{Bool}(undef, 0, size(train_targets, 2)) # Inicializar validation targets como matriz vacía

        if validationRatio > 0 # En caso de que tengamos validación

            v_ratio = validationRatio * (folds/(folds-1)) # Calcular ratio de validación adaptado.

            trainIndices, validationIndices = holdOut(size(train_inputs, 1), v_ratio) # Calcular indices de validación
            validation_inputs = train_inputs[validationIndices, :] # Extraer inputs de validación
            validation_targets = train_targets[validationIndices, :] # Extraer targets de validación
            train_inputs = train_inputs[trainIndices, :] # Extraer inputs de entrenamiento
            train_targets = train_targets[trainIndices, :] # Extraer targets de entrenamiento
        end

        # Crear nuevos vectores para las métricas de cada epoch de cada fold.
        
        acc_folf = []
        fail_rate_fold = []
        recall_fold = []
        especificity_fold = []
        precision_fold = []
        npv_fold = []
        f1_fold = []
        cnf_matrix_fold = Array{Float64}(undef, length(classes), length(classes), numExecutions)

        # Entrenar fold
        for i in 1:numExecutions

            # Entrenar ANN
            ann, _, _, _ = trainClassANN(
                topology, 
                (train_inputs, train_targets);
                validationDataset=(validation_inputs, validation_targets),
                testDataset=(test_inputs, test_targets),
                transferFunctions=transferFunctions, 
                maxEpochs=maxEpochs, 
                minLoss=minLoss, 
                learningRate=learningRate)

            # Calcular métricas
            metrics = confusionMatrix(ann(test_inputs')', test_targets)

            # Añadir métricas al registro.
            push!(acc_folf, metrics[1])
            push!(fail_rate_fold, metrics[2])
            push!(recall_fold, metrics[3])
            push!(especificity_fold, metrics[4])
            push!(precision_fold, metrics[5])
            push!(npv_fold, metrics[6])
            push!(f1_fold, metrics[7])
            cnf_matrix_fold[:, :, i] = metrics[8]

        end

        # Calcular métricas de cada fold
        push!(accuracy, mean(acc_folf))
        push!(fail_rate, mean(fail_rate_fold))
        push!(recall, mean(recall_fold))
        push!(especificity, mean(especificity_fold))
        push!(precision, mean(precision_fold))
        push!(npv, mean(npv_fold))
        push!(f1, mean(f1_fold))
        confussion_matrix += dropdims(mean(cnf_matrix_fold, dims=3), dims=3)
    end

    # Devolver media y desviación de las métricas
    return ((mean(accuracy), std(accuracy)), (mean(fail_rate), std(fail_rate)), (mean(recall), std(recall)), (mean(especificity), std(especificity)), (mean(precision), std(precision)), (mean(npv), std(npv)), (mean(f1), std(f1)), confussion_matrix)

end

ANNCrossValidation (generic function with 1 method)

## Implement testing pipeline

In [31]:
ruta_rafa = "/Users/rafa/Documents/Uni/Cuatri4/machine_learning/FAA_practices/iris.data"

dataset = readdlm(ruta_rafa,',')

# Preparamos las entradas
inputs = Float32.(dataset[:,1:4])
targets = dataset[:,5]
inputs = normalizeZeroMean!(inputs)
kf_indices = crossvalidation(targets, 10)   

150-element Vector{Int64}:
  1
  7
  8
  6
  5
  2
  3
  3
  7
  5
  ⋮
  6
  9
  9
  6
  3
 10
 10
  1
  5

In [32]:
insights = ANNCrossValidation(
    [8, 6], 
    (inputs, targets),
    kf_indices;
    numExecutions=100,
    transferFunctions=[σ, σ],
    maxEpochs=100,
    minLoss=0.0,
    learningRate=0.01,
    validationRatio=0.2,
    maxEpochsVal=10
)

((0.8704666666666661, 0.06234753254792693), (0.12953333333333317, 0.062347532547926944), (0.9136444444444436, 0.04156502169861767), (0.08635555555555549, 0.041565021698617956), (0.8704666666666661, 0.06234753254792693), (0.9352333333333338, 0.031173766273963736), (0.9127526455026445, 0.028101979982825237), [47.99 0.09000000000000001 0.05; 0.61 37.830000000000005 5.200000000000001; 1.4000000000000001 12.079999999999998 44.74999999999999])

# Ejercicio 6

In [2]:
import Pkg
Pkg.add("MLJ")

    Updating registry at `~/.julia/registries/General.toml`
   Resolving package versions...
   Installed HypergeometricFunctions ── v0.3.27
   Installed PDMats ─────────────────── v0.11.32
   Installed StatisticalTraits ──────── v3.4.0
   Installed CategoricalDistributions ─ v0.1.15
   Installed EarlyStopping ──────────── v0.3.0
   Installed StatsFuns ──────────────── v1.3.2
   Installed IterationControl ───────── v0.5.4
   Installed LearnAPI ───────────────── v0.1.0
   Installed StatisticalMeasures ────── v0.1.7
   Installed MLJFlow ────────────────── v0.2.0
   Installed Rmath_jll ──────────────── v0.5.1+0
   Installed PrettyPrinting ─────────── v0.4.2
   Installed MLJIteration ───────────── v0.6.3
   Installed MLJBase ────────────────── v1.7.0
   Installed MLJModels ──────────────── v0.16.17
   Installed ScientificTypes ────────── v3.1.0
   Installed CategoricalArrays ──────── v0.10.8
   Installed StatisticalMeasuresBase ── v0.1.2
   Installed ARFFFiles ──────────────── v1.5.0
   In

In [7]:
using MLJ

In [5]:
Pkg.add("LIBSVM")
Pkg.add("NearestNeighborModels")

Pkg.add("MLJLIBSVMInterface")
Pkg.add("MLJDecisionTreeInterface")

   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
  No Changes to `~/.julia/environments/v1.11/Project.toml`
  No Changes to `~/.julia/environments/v1.11/Manifest.toml`
   Resolving package versions...
   Installed MLJLIBSVMInterface ─ v0.2.1
    Updating `~/.julia/environments/v1.11/Project.toml`
  [61c7150f] + MLJLIBSVMInterface v0.2.1
    Updating `~/.julia/environments/v1.11/Manifest.toml`
  [61c7150f] + MLJLIBSVMInterface v0.2.1
Precompiling project...
   4002.5 ms  ✓ MLJLIBSVMInterface
  1 dependency successfully precompiled in 6 seconds. 502 already precompiled.
   Resolving package versions...
   Installed DecisionTree ───────────── v0.12.4
   Installed MLJDecisionTreeInterface ─ v0.4.2
   Installed AbstractTrees ──────────── v0.4.5
    Updating `~/.julia/environments/v1.11/Project.toml`
  [c6f25543] + MLJDecisionTreeInterface v0.4.2
    Updatin

In [154]:
using LIBSVM

In [150]:
SVMClassifier = MLJ.@load SVC pkg=LIBSVM verbosity=0
kNNClassifier = MLJ.@load KNNClassifier pkg=NearestNeighborModels verbosity=0
DTClassifier = MLJ.@load DecisionTreeClassifier pkg=DecisionTree verbosity=0 

MLJDecisionTreeInterface.DecisionTreeClassifier

In [174]:
function modelCrossValidation(modelType::Symbol, modelHyperparameters::Dict,
    dataset::Tuple{AbstractArray{<:Real,2}, AbstractArray{<:Any,1}},
    crossValidationIndices::Array{Int64,1})

    
    if modelType == :ANN
        # Comprobar que existe el parámetro obligatorio
        @assert(haskey(modelHyperparameters, "topology"), "El parámetro 'topology' es obligatorio para ANN")
        @assert(isa(modelHyperparameters["topology"], AbstractArray{<:Int,1}), 
                "topology debe ser un AbstractArray{<:Int,1}")

        # Crear un nuevo diccionario para los parámetros opcionales
        ann_params = Dict{Symbol, Any}()
        
        # Validar y añadir cada parámetro opcional si existe
        if haskey(modelHyperparameters, "numExecutions")
            @assert(isa(modelHyperparameters["numExecutions"], Int), 
                    "numExecutions debe ser de tipo Int")
            ann_params[:numExecutions] = modelHyperparameters["numExecutions"]
        end
        
        if haskey(modelHyperparameters, "transferFunctions")
            @assert(isa(modelHyperparameters["transferFunctions"], AbstractArray{<:Function,1}), 
                    "transferFunctions debe ser un AbstractArray{<:Function,1}")
            ann_params[:transferFunctions] = modelHyperparameters["transferFunctions"]
        end
        
        if haskey(modelHyperparameters, "maxEpochs")
            @assert(isa(modelHyperparameters["maxEpochs"], Int), 
                    "maxEpochs debe ser de tipo Int")
            ann_params[:maxEpochs] = modelHyperparameters["maxEpochs"]
        end
        
        if haskey(modelHyperparameters, "minLoss")
            @assert(isa(modelHyperparameters["minLoss"], Real), 
                    "minLoss debe ser de tipo Real")
            ann_params[:minLoss] = modelHyperparameters["minLoss"]
        end
        
        if haskey(modelHyperparameters, "learningRate")
            @assert(isa(modelHyperparameters["learningRate"], Real), 
                    "learningRate debe ser de tipo Real")
            ann_params[:learningRate] = modelHyperparameters["learningRate"]
        end
        
        if haskey(modelHyperparameters, "validationRatio")
            @assert(isa(modelHyperparameters["validationRatio"], Real), 
                    "validationRatio debe ser de tipo Real")
            ann_params[:validationRatio] = modelHyperparameters["validationRatio"]
        end
        
        if haskey(modelHyperparameters, "maxEpochsVal")
            @assert(isa(modelHyperparameters["maxEpochsVal"], Int), 
                    "maxEpochsVal debe ser de tipo Int")
            ann_params[:maxEpochsVal] = modelHyperparameters["maxEpochsVal"]
        end

        # Llamar a ANNCrossValidation con los argumentos validados
        return ANNCrossValidation(
            modelHyperparameters["topology"], 
            dataset, 
            crossValidationIndices; 
            pairs(ann_params)...
        )

    else

        inputs, targets = dataset # descomponer dataset
        targets = string.(targets) # convertir a string
        classes = unique(targets) # calcular las clases
        folds = maximum(crossValidationIndices) # Calcular el número de folds

        # Reservar espacio para las métricas
        accuracy = Float64[] # Vector de precisión (accuracy)
        fail_rate = Float64[] # Vector de tasa de error (error rate)
        recall = Float64[] # Vector de sensibilidad (recall)
        especificity = Float64[] # Vector de especificidad
        precision = Float64[] # Vector de VPP (precisión)
        npv = Float64[] # Vector de VPN
        f1 = Float64[] # Vector de F1
        confussion_matrix = zeros(length(classes), length(classes)) # Inicializar confussion matrix
    

        for fold in 1:folds

            train_inputs = inputs[findall(crossValidationIndices .!= fold), :]
            train_targets = targets[findall(crossValidationIndices .!= fold), :]
            test_inputs = inputs[findall(crossValidationIndices .== fold), :]
            test_targets = targets[findall(crossValidationIndices .== fold), :]

            if modelType == :DoME # DoME

                @assert(haskey(modelHyperparameters, "maximumNodes")) # Asegurarse que tenga hipermarámetro obligatiorio
                @assert(isa(modelHyperparameters["maximumNodes"], Int))

                model_output = trainClassDoME((train_inputs, train_targets[:]), test_inputs, modelHyperparameters["maximumNodes"])
    

            elseif modelType == :SVC # SVM

                # Crear un nuevo diccionario para los parámetros de SVM
                svm_params = Dict{Symbol, Any}()
                
                # Verificar el parámetro C (obligatorio para todos los kernels)
                @assert(haskey(modelHyperparameters, "C"), "El parámetro 'C' es obligatorio para SVM")
                @assert(isa(modelHyperparameters["C"], Real), "C debe ser un número Real")
                C = Float64(modelHyperparameters["C"])
                
                # Verificar el parámetro kernel (obligatorio)
                @assert(haskey(modelHyperparameters, "kernel"), "El parámetro 'kernel' es obligatorio para SVM")
                @assert(isa(modelHyperparameters["kernel"], String), "kernel debe ser un String")
                
                # Validar el tipo de kernel y sus parámetros correspondientes
                kernel_type = lowercase(modelHyperparameters["kernel"])

                kernel = LIBSVM.Kernel.Linear
                
                if kernel_type == "linear"
                    # Para kernel lineal solo se necesita C, que ya se verificó
                    kernel = LIBSVM.Kernel.Linear
                    
                elseif kernel_type == "rbf" || kernel_type == "radialbasis"
                    # Para kernel RBF se necesita gamma además de C
                    @assert(haskey(modelHyperparameters, "gamma"), "El parámetro 'gamma' es obligatorio para kernel RBF")
                    @assert(isa(modelHyperparameters["gamma"], Real), "gamma debe ser un número Real")
                    kernel = LIBSVM.Kernel.RadialBasis
                    svm_params[:gamma] = Float64(modelHyperparameters["gamma"])
                    
                elseif kernel_type == "sigmoid"
                    # Para kernel sigmoidal se necesitan gamma y coef0 además de C
                    @assert(haskey(modelHyperparameters, "gamma"), "El parámetro 'gamma' es obligatorio para kernel Sigmoid")
                    @assert(isa(modelHyperparameters["gamma"], Real), "gamma debe ser un número Real")
                    svm_params[:gamma] = Float64(modelHyperparameters["gamma"])
                    
                    @assert(haskey(modelHyperparameters, "coef0"), "El parámetro 'coef0' es obligatorio para kernel Sigmoid")
                    @assert(isa(modelHyperparameters["coef0"], Real), "coef0 debe ser un número Real")
                    svm_params[:coef0] = Float64(modelHyperparameters["coef0"])
                    
                    kernel = LIBSVM.Kernel.Sigmoid
                    
                elseif kernel_type == "polynomial" || kernel_type == "poly"
                    # Para kernel polinómico se necesitan gamma, coef0 y degree además de C
                    @assert(haskey(modelHyperparameters, "gamma"), "El parámetro 'gamma' es obligatorio para kernel Polynomial")
                    @assert(isa(modelHyperparameters["gamma"], Real), "gamma debe ser un número Real")
                    svm_params[:gamma] = Float64(modelHyperparameters["gamma"])
                    
                    @assert(haskey(modelHyperparameters, "coef0"), "El parámetro 'coef0' es obligatorio para kernel Polynomial")
                    @assert(isa(modelHyperparameters["coef0"], Real), "coef0 debe ser un número Real")
                    svm_params[:coef0] = Float64(modelHyperparameters["coef0"])
                    
                    @assert(haskey(modelHyperparameters, "degree"), "El parámetro 'degree' es obligatorio para kernel Polynomial")
                    @assert(isa(modelHyperparameters["degree"], Real), "degree debe ser un número Real")
                    svm_params[:degree] = Int32(modelHyperparameters["degree"])
                    
                    kernel = LIBSVM.Kernel.Polynomial
                    
                else
                    error("Tipo de kernel no soportado: $(modelHyperparameters["kernel"]). Los valores aceptados son: linear, rbf, sigmoid, polynomial")
                end
                
                # Crear el modelo SVM
                model = SVMClassifier(kernel=kernel, cost=C; svm_params...)
                mach = machine(model, MLJ.table(train_inputs), categorical(train_targets[:])) # crear objeto modelo
                MLJ.fit!(mach, verbosity=0) # Ajustar modelo
                model_output = MLJ.predict(mach, MLJ.table(test_inputs)) # Ejecutar sobre el conjunto de test
    
            elseif modelType == :DecisionTreeClassifier # DecisionTree
    
                @assert(haskey(modelHyperparameters, "max_depth")) # Asegurarse que tenga hipermarámetro obligatiorio
                @assert(haskey(modelHyperparameters, "rng"))

                model = DTClassifier(max_depth=modelHyperparameters["max_depth"], rng=modelHyperparameters["rng"]) # Crear el modelo
                mach = machine(model, MLJ.table(train_inputs), categorical(train_targets[:])) # crear objeto modelo
                MLJ.fit!(mach, verbosity=0) # Ajustar modelo
                output = MLJ.predict(mach, MLJ.table(test_inputs)) # Ejecutar sobre el conjunto de test
                model_output = mode.(output) # Convertir a vector categórico


            elseif modelType == :KNNClassifier # KNN
    
                @assert(haskey(modelHyperparameters, "k"))

                model = kNNClassifier(K = modelHyperparameters["k"]) # Crear el modelo
                mach = machine(model, MLJ.table(train_inputs), categorical(train_targets[:])) # crear objeto modelo
                MLJ.fit!(mach, verbosity=0) # Ajustar modelo
                output = MLJ.predict(mach, MLJ.table(test_inputs)) # Ejecutar sobre el conjunto de test
                model_output = mode.(output) # Convertir a vector categórico
    
            else
                println("Unexpected model type")
                
            end

            # Calcular métricas del fold
            metrics = confusionMatrix(model_output, test_targets[:], classes)
            push!(accuracy, metrics[1])
            push!(fail_rate, metrics[2])
            push!(recall, metrics[3])
            push!(especificity, metrics[4])
            push!(precision, metrics[5])
            push!(npv, metrics[6])
            push!(f1, metrics[7])
            confussion_matrix += metrics[8]
            
        end
    end

    # Devolver media y desviación de las métricas
    return (
            (mean(accuracy), std(accuracy)),
            (mean(fail_rate), std(fail_rate)),
            (mean(recall), std(recall)),
            (mean(especificity), std(especificity)),
            (mean(precision), std(precision)),
            (mean(npv), std(npv)),
            (mean(f1), std(f1)),
            confussion_matrix
           )

end


modelCrossValidation (generic function with 1 method)

In [170]:
ruta_rafa = "/Users/rafa/Documents/Uni/Cuatri4/machine_learning/FAA_practices/iris.data"

dataset = readdlm(ruta_rafa,',')

# Preparamos las entradas
inputs = Float32.(dataset[:,1:4])
targets = dataset[:,5]
inputs = normalizeZeroMean!(inputs)
kf_indices = crossvalidation(targets, 10)   

150-element Vector{Int64}:
  8
  2
  8
  9
  5
 10
  1
  5
  1
  6
  ⋮
  4
  5
  1
  6
 10
  8
  2
  6
  3

In [188]:
rng = MersenneTwister(42) 

modelHyperparameters = Dict("kernel" => "polynomial", "C"=>0.001, "gamma"=>0.1, "coef0"=>1, "degree"=>6);

In [189]:
modelCrossValidation(:SVC, modelHyperparameters, (inputs, targets), kf_indices)

((0.6199999999999999, 0.10446808243149473), (0.38, 0.10446808243149473), (0.7466666666666666, 0.06964538828766315), (0.2533333333333333, 0.06964538828766316), (0.6199999999999999, 0.10446808243149472), (0.8099999999999999, 0.05223404121574735), (0.8255147630147629, 0.023768709412357498), [35.0 0.0 0.0; 15.0 50.0 42.0; 0.0 0.0 8.0])